# 5G Network Intelligence - Historical Exceedance Probability

## Hypothesis and prior evidence

The MAE-focused MLP blend has low recall for upper historical exceedances. Test direct binary probability prediction rather than treating a median-like point forecast as a probability. No improvement or operational congestion detection is assumed.

## Temporal protocol

Fit/stop/score boundaries remain 335/383/443 and 395/443/503. Construct each fold's per-beam P90/P95 from its fit prefix only, then freeze them for stopping/scoring. The saved hour-503 validation thresholds must not label earlier development windows. All eligible observed targets and original beam axes are retained. Reused development is not fresh-test evidence. Validation 504-671, final holdout and production are excluded.


In [ ]:
RUN_TRAINING = False
SOURCE_COMMIT = 'ec38b37cbc63dabf382c017bfe77654e3fbc7f71'


## Fixed model and metrics

Twelve LightGBM binary fits: two folds, three horizons, two historical thresholds. Keep 40 causal features, no class reweighting or calibration. Thresholds define labels and scoring; fitted beam summaries are not model inputs. Primary metrics are Brier, log loss and average precision. A fit-prevalence control uses no future labels. Report precision/recall/F1 at fixed probability 0.5; do not choose a threshold from scoring results. Probabilities do not use DLPRB units and do not replace the MAE/WAPE target.


In [ ]:
PROTOCOL = {'protocol': 'direct-historical-exceedance-probability-v1',
 'folds': [{'name': 'inner1', 'fit_end': 335, 'stop_end': 383, 'score_end': 443},
           {'name': 'inner2', 'fit_end': 395, 'stop_end': 443, 'score_end': 503}],
 'horizons': [1, 6, 24],
 'thresholds': ['warning', 'critical'],
 'max_rounds': 800,
 'patience': 60,
 'stride': 1,
 'seed': 17,
 'threads': 4,
 'params': {'num_leaves': 63,
            'learning_rate': 0.035,
            'min_data_in_leaf': 100,
            'lambda_l2': 0.5},
 'early_stopping_metric': 'binary_logloss',
 'decision_threshold': 0.5,
 'class_reweighting': 'forbidden',
 'probability_calibration': 'none',
 'input_contract': 'original_40_causal_features_no_fitted_beam_summaries',
 'historical_threshold_policy': 'configs/risk.yaml_each_fold_fit_prefix_only',
 'primary_metrics': ['brier', 'log_loss', 'average_precision'],
 'secondary_metrics': 'fixed_0.5_precision_recall_F1',
 'validation_504_671': 'forbidden',
 'original_final_holdout': 'forbidden',
 'production_change': 'forbidden',
 'claim_scope': 'one_seed_reused_development_only_not_operational_congestion'}


## Environment and source

Attach the completed original hurdle worker for canonical data and inherited causal feature code. A separate private MIT source dataset supplies the new trainer, probability metrics and pinned configurations. Full source remains in named modules.


In [ ]:
from pathlib import Path
import os,sys,shutil,subprocess,tarfile
if RUN_TRAINING:
    if not Path('/kaggle/working').is_dir():
        raise RuntimeError('Hosted Kaggle required')
    roots=[p.parent.parent for p in Path('/kaggle/input').rglob('hurdle_development_scores.csv') if (p.parent.parent/'data/processed/panel.npz').exists()]
    if len(roots)!=1:
        raise RuntimeError('Attach exactly one completed canonical source')
    PROJECT=Path('/kaggle/working/risk_probability')
    PROJECT.mkdir(exist_ok=True)
    for folder in ('src','scripts','configs','data/processed'):
        shutil.copytree(roots[0]/folder,PROJECT/folder,dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
    os.chdir(PROJECT)
    sys.path.insert(0,str(PROJECT))
    os.environ['SOURCE_GIT_COMMIT']=SOURCE_COMMIT
    patches=list(Path('/kaggle/input').rglob('train_risk_probability.py'))
    if len(patches)!=1:
        raise RuntimeError('Attach exactly one probability study source')
    patch=patches[0].parent
    for name,folder in (('train_risk_probability.py','scripts'),('probability_metrics.py','src/risk'),('risk_probability.yaml','configs'),('risk.yaml','configs')):
        shutil.copyfile(patch/name,Path(folder)/name)
    Path('reports').mkdir(exist_ok=True)


## Hosted training and debugging

Save fold thresholds, binary checkpoints, stopping rounds, score arrays and provenance. Preserve missing targets as missing labels. Save outputs after every classifier; a failure must retain logs rather than report success.


In [ ]:
if RUN_TRAINING:
    with Path('reports/risk_probability_training.log').open('w') as log:
        process=subprocess.Popen([sys.executable,'-u','-m','scripts.train_risk_probability'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True)
            log.write(line)
            log.flush()
        status=process.wait()
    with tarfile.open('/kaggle/working/risk_probability_results.tar.gz','w:gz') as archive:
        for folder in ('reports','artifacts/risk_probability'):
            if Path(folder).exists():
                archive.add(folder,arcname=folder)
    if status:
        raise RuntimeError(f'Hosted probability study failed: {status}; inspect saved logs')
else:
    print('Review only: no fitting or calibration.')


## Results and limitations

Report both thresholds at every horizon/fold, eligible counts and undefined metrics. Better ranking or Brier score does not imply calibrated deployment probabilities or a useful operating point. Historical quantiles are not operator limits or congestion labels. A later unseen-data study is required before any production change.


In [ ]:
paths=[Path('reports/risk_probability_development.csv'),Path('../reports/risk_probability_development.csv')]
if Path('/kaggle/input').is_dir():
    paths+=list(Path('/kaggle/input').rglob('risk_probability_development.csv'))
path=next((p for p in paths if p.is_file()),None)
if path:
    import pandas as pd
    from IPython.display import display
    display(pd.read_csv(path))
else:
    print('No completed probability report attached.')
